# BindCraft Binder Design — One-Off Sampling Example

![BindCraft Binder Design — One-Off Sampling Example](https://proto-bio.github.io/proto-assets/images/tool/bindcraft/hero.png)

Design a single protein binder against a target with the BindCraft pipeline
(AlphaFold2 hallucination + ProteinMPNN refinement + PyRosetta filtering).

- Paper: [Pacesa et al., Nature 2025](https://www.nature.com/articles/s41586-025-09429-6)
- Upstream repo (pinned commit `7cd4ace`): https://github.com/martinpacesa/BindCraft/tree/7cd4ace1b7407adf66a50dfefa47de2270f5e4a9

This example uses tiny iteration counts and `max_trajectories=1` for a
~10-30 minute run on an H100. For production runs (100 binders, 24-72h),
leave `BindCraftConfig` at its upstream defaults and set
`number_of_final_designs=100` on the input.

> **Note: long-running tool.** Binder design campaigns take a long time on GPU, so the design cells below are left unexecuted and shown for reference.

In [1]:
from proto_tools.tools.binder_design.bindcraft import (
    BindCraftConfig,
    BindCraftInput,
    run_bindcraft_design,
)
from proto_tools.utils.notebook_docs import display_api_reference


## Input API Reference

In [2]:
display_api_reference("bindcraft", "input")

**Input** — `BindCraftInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_pdb</code> | <code>Structure</code> | required | Target structure. |
| <code>target_chain</code> | <code>str</code> | <code>'A'</code> | Chain ID(s) of the frozen target (comma-separated for multi-chain). |
| <code>target_hotspot_residues</code> | <code>str &#124; None</code> | <code>None</code> | Comma-separated 1-indexed residue positions on the target (e.g. '1-10,56,78'). |
| <code>binder_lengths</code> | <code>tuple[int, int]</code> | <code>(65, 150)</code> | (min, max) binder length range. Upstream default: (65, 150). |
| <code>binder_name</code> | <code>str</code> | <code>'binder'</code> | Filename prefix for accepted designs (e.g. 'binder' yields 'binder_l60_s12345_mpnn3'). |
| <code>number_of_final_designs</code> | <code>int</code> | <code>100</code> | Target accepted-design count. Set to 1 for one-off sampling. Upstream default: 100. |

## Config API Reference

In [3]:
display_api_reference("bindcraft", "config")

**Config** — `BindCraftConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the tool on. |
| <code>timeout</code> | <code>int &#124; None</code> | <code>None</code> | Maximum execution time in seconds. None (default) waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>design_algorithm</code> | <code>Literal['2stage', '3stage', '4stage', 'greedy', 'mcmc']</code> | <code>'4stage'</code> | Hallucination algorithm; '4stage' runs soft, temporary, hard, then greedy stages. |
| <code>use_multimer_design</code> | <code>bool</code> | <code>True</code> | Use AF2 multimer params during hallucination. Every upstream preset uses multimer. |
| <code>omit_AAs</code> | <code>str</code> | <code>'C'</code> | Comma-separated amino acids to ban during design (e.g. 'C' or 'C,W'). |
| <code>force_reject_AA</code> | <code>bool</code> | <code>False</code> | Drop any MPNN sequence containing residues from omit_AAs (hard reject, not a soft penalty). |
| <code>soft_iterations</code> | <code>int</code> | <code>75</code> | Soft-stage hallucination iterations. Used by 2stage / 3stage / 4stage. |
| <code>temporary_iterations</code> | <code>int</code> | <code>45</code> | Temporary-stage hallucination iterations. Used by 3stage / 4stage. |
| <code>hard_iterations</code> | <code>int</code> | <code>5</code> | Hard-stage hallucination iterations. Used by 3stage / 4stage. |
| <code>greedy_iterations</code> | <code>int</code> | <code>15</code> | Greedy-stage iteration count. Used by 2stage / 4stage / greedy / mcmc (3stage doesn't use it). |
| <code>greedy_percentage</code> | <code>float</code> | <code>1.0</code> | Mutation rate for greedy/MCMC tries (% of binder length). Used by 2stage / 4stage / greedy / mcmc. |
| <code>weights_plddt</code> | <code>float</code> | <code>0.1</code> | Loss weight on AF2 pLDDT (higher = push for more confident-folding designs). |
| <code>weights_pae_intra</code> | <code>float</code> | <code>0.4</code> | Loss weight on within-binder PAE (higher = push for cleaner internal geometry). |
| <code>weights_pae_inter</code> | <code>float</code> | <code>0.1</code> | Loss weight on the binder-target interface PAE; higher values push for more confident pairing. |
| <code>weights_con_intra</code> | <code>float</code> | <code>1.0</code> | Loss weight on within-binder C-alpha contacts (encourages a compact fold). |
| <code>weights_con_inter</code> | <code>float</code> | <code>1.0</code> | Loss weight on the binder-target interface contacts; encourages docking. |
| <code>weights_helicity</code> | <code>float</code> | <code>-0.3</code> | Helicity bias weight (negative discourages helices, positive encourages them). |
| <code>weights_iptm</code> | <code>float</code> | <code>0.05</code> | Loss weight on AF2 interface pTM (higher = push for higher iPTM). |
| <code>weights_rg</code> | <code>float</code> | <code>0.3</code> | Loss weight on binder radius of gyration (higher = compress toward a tighter binder). |
| <code>weights_termini_loss</code> | <code>float</code> | <code>0.1</code> | Loss weight pulling binder N- and C-termini together (for cyclisable backbones). |
| <code>random_helicity</code> | <code>bool</code> | <code>False</code> | Randomise the sign of weights_helicity per trajectory. |
| <code>use_i_ptm_loss</code> | <code>bool</code> | <code>True</code> | Enable interface pTM loss term (weights_iptm). Pushes for higher iPTM during hallucination. |
| <code>use_rg_loss</code> | <code>bool</code> | <code>True</code> | Enable radius-of-gyration loss. |
| <code>use_termini_distance_loss</code> | <code>bool</code> | <code>False</code> | Enable N-/C-termini distance loss. |
| <code>intra_contact_distance</code> | <code>float</code> | <code>14.0</code> | Intra-chain contact distance cutoff (Å). |
| <code>inter_contact_distance</code> | <code>float</code> | <code>20.0</code> | Inter-chain (interface) contact distance cutoff (Å). |
| <code>intra_contact_number</code> | <code>int</code> | <code>2</code> | Number of intra-chain contacts per residue. |
| <code>inter_contact_number</code> | <code>int</code> | <code>2</code> | Number of inter-chain contacts per residue. |
| <code>rm_template_seq_design</code> | <code>bool</code> | <code>False</code> | Hide target sequence from AF2 template during hallucination (geometry-only target). |
| <code>rm_template_seq_predict</code> | <code>bool</code> | <code>False</code> | Hide target sequence from AF2 template during validation (geometry-only target). |
| <code>rm_template_sc_design</code> | <code>bool</code> | <code>False</code> | Hide target side-chain coordinates from AF2 template during hallucination. |
| <code>rm_template_sc_predict</code> | <code>bool</code> | <code>False</code> | Hide target side-chain coordinates from AF2 template during validation. |
| <code>predict_initial_guess</code> | <code>bool</code> | <code>False</code> | Seed AF2 validation with the hallucinated complex coords (helps hard targets). |
| <code>predict_bigbang</code> | <code>bool</code> | <code>False</code> | Seed AF2 validation with the hallucinated atom positions (alternative to initial_guess). |
| <code>enable_mpnn</code> | <code>bool</code> | <code>True</code> | Refine each hallucinated binder with ProteinMPNN before AF2 re-validation. |
| <code>mpnn_fix_interface</code> | <code>bool</code> | <code>True</code> | Fix interface residues during MPNN redesign. |
| <code>num_seqs</code> | <code>int</code> | <code>20</code> | MPNN sequences sampled per accepted trajectory before filtering. |
| <code>max_mpnn_sequences</code> | <code>int</code> | <code>2</code> | Top-scoring MPNN sequences (out of num_seqs) carried forward to AF2 validation. |
| <code>sampling_temp</code> | <code>float</code> | <code>0.1</code> | MPNN sampling temperature (lower = more deterministic). |
| <code>backbone_noise</code> | <code>float</code> | <code>0.0</code> | Std-dev of Gaussian noise added to backbone coords before MPNN sampling (0 = none). |
| <code>model_path</code> | <code>Literal['v_48_002', 'v_48_010', 'v_48_020', 'v_48_030']</code> | <code>'v_48_020'</code> | MPNN checkpoint name. Higher trailing digits = trained with more backbone noise. |
| <code>mpnn_weights</code> | <code>Literal['original', 'soluble']</code> | <code>'soluble'</code> | MPNN weight set. 'soluble' biases toward soluble residues; 'original' is the unbiased release. |
| <code>num_recycles_design</code> | <code>int</code> | <code>1</code> | AlphaFold2 recycles during hallucination. |
| <code>num_recycles_validation</code> | <code>int</code> | <code>3</code> | AlphaFold2 recycles during validation. |
| <code>optimise_beta</code> | <code>bool</code> | <code>True</code> | When a trajectory looks β-heavy (>15% sheet), bump iterations + recycles to help it converge. |
| <code>optimise_beta_extra_soft</code> | <code>int</code> | <code>0</code> | Extra soft iterations for beta-heavy designs (4stage only — added after soft stage). |
| <code>optimise_beta_extra_temp</code> | <code>int</code> | <code>0</code> | Extra temporary iterations for beta-heavy designs (4stage only). |
| <code>optimise_beta_recycles_design</code> | <code>int</code> | <code>3</code> | AF2 recycles during hallucination for beta-heavy designs. |
| <code>optimise_beta_recycles_valid</code> | <code>int</code> | <code>3</code> | AF2 recycles during validation for beta-heavy designs. |
| <code>max_trajectories</code> | <code>int &#124; bool</code> | <code>False</code> | Max hallucination trajectories before stopping. False = unlimited; positive int = cap. |
| <code>enable_rejection_check</code> | <code>bool</code> | <code>True</code> | Enable rolling acceptance-rate monitoring (stops the run if it stalls). |
| <code>acceptance_rate</code> | <code>float</code> | <code>0.01</code> | Minimum design acceptance rate to keep running. |
| <code>start_monitoring</code> | <code>int</code> | <code>600</code> | Trajectory count before acceptance-rate monitoring starts. |
| <code>filter_overrides</code> | <code>dict[str, Any]</code> | <code>{}</code> | Per-metric threshold overrides merged on top of upstream default_filters.json. |

## Output API Reference

In [4]:
display_api_reference("bindcraft", "output")

**Output** — `BindCraftOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>designs</code> | <code>list[BindCraftDesign]</code> | <code>[]</code> | Accepted binder designs. |
| <code>n_trajectories_run</code> | <code>int</code> | <code>0</code> | Total trajectories attempted before stopping. |
| <code>n_designs_accepted</code> | <code>int</code> | <code>0</code> | Designs that passed all filters (equals len(designs)). |

**`BindCraftDesign`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>design_name</code> | <code>str</code> | required | Unique design identifier emitted by BindCraft. |
| <code>binder_sequence</code> | <code>str</code> | required | Designed binder amino-acid sequence (1-letter codes). |
| <code>structure</code> | <code>Structure</code> | required | Relaxed target+binder complex with per-residue pLDDT in B-factors. |
| <code>metrics</code> | <code>BindCraftMetrics</code> | required | Per-design averaged metrics evaluated by the filter check. |
| <code>seed</code> | <code>int</code> | required | Random seed of the trajectory that produced this design. |
| <code>interface_aas</code> | <code>dict[str, int]</code> | <code>{}</code> | Amino-acid composition at the binder-target interface. |
| <code>interface_residues</code> | <code>list[int]</code> | <code>[]</code> | 1-indexed binder residue positions at the interface. |

**Metrics** — `BindCraftMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>avg_plddt</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_ptm</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_iptm</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>avg_ipae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>avg_iplddt</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_ss_plddt</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_binder_plddt</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_binder_ptm</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>avg_binder_pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>binder_energy_score</code> | <code>float</code> |  | <code>REU</code> |  |
| <code>dG</code> | <code>float</code> |  | <code>REU</code> |  |
| <code>dSASA</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å^2</code> |  |
| <code>dG_per_dSASA</code> | <code>float</code> |  | <code>REU/Å^2</code> |  |
| <code>interface_sasa_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>interface_hydrophobicity</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>surface_hydrophobicity</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>shape_complementarity</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>packstat</code> | <code>float</code> | <code>[0, 1]</code> | <code>fraction</code> |  |
| <code>n_interface_hbonds</code> | <code>float</code> | <code>&gt;= 0</code> | <code>count</code> |  |
| <code>interface_hbonds_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>n_interface_unsat_hbonds</code> | <code>float</code> | <code>&gt;= 0</code> | <code>count</code> |  |
| <code>interface_unsat_hbonds_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>n_interface_residues</code> | <code>float</code> | <code>&gt;= 0</code> | <code>count</code> |  |
| <code>binder_helix_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>binder_betasheet_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>binder_loop_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>interface_helix_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>interface_betasheet_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>interface_loop_pct</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> |  |
| <code>hotspot_rmsd</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>target_rmsd</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>binder_rmsd</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>unrelaxed_clashes</code> | <code>float</code> | <code>&gt;= 0</code> | <code>count</code> |  |
| <code>relaxed_clashes</code> | <code>float</code> | <code>&gt;= 0</code> | <code>count</code> |  |

In [ ]:
from pathlib import Path

# Resolve the in-tree renin test target by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pdb_path = _repo_root / "tests" / "dummy_data" / "renin_af3.pdb"
print(f"Using target PDB: {pdb_path}")

inputs = BindCraftInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    target_hotspot_residues="56",          # 1-indexed residue position(s) on the target
    binder_lengths=(60, 70),               # (min, max)
    binder_name="example",
    number_of_final_designs=1,             # one-off sampling
)

config = BindCraftConfig(
    max_trajectories=1,                    # cap at 1 trajectory
    soft_iterations=10,                    # tiny iteration budget
    temporary_iterations=5,
    hard_iterations=2,
    greedy_iterations=2,
    num_seqs=2,
    max_mpnn_sequences=1,
    seed=42,                               # reproducible
)

result = run_bindcraft_design(inputs, config)
print(f"Trajectories run: {result.n_trajectories_run}")
print(f"Designs accepted: {result.n_designs_accepted}")

In [ ]:
if result.designs:
    design = result.designs[0]
    print(f"Design: {design.design_name}")
    print(f"Sequence ({len(design.binder_sequence)} aa): {design.binder_sequence}")
    print(f"Seed: {design.seed}")
    print(f"avg pLDDT: {design.metrics['avg_plddt']:.3f}")
    print(f"avg iPTM:  {design.metrics['avg_iptm']:.3f}")
    print(f"dG:        {design.metrics['dG']:.2f} REU")
    print(f"Shape complementarity: {design.metrics['shape_complementarity']:.3f}")
else:
    print("No designs passed the filter check — try more trajectories or relaxed filters.")

## Tuning runtime

BindCraft is a stochastic, filter-driven pipeline — most trajectories fail the filter check, so the wall-clock time depends on both the number of accepted designs you want and how strict the filters are. Three useful operating points:

- **One-off sampling (~10-30 min on H100)**: this notebook — `number_of_final_designs=1`, `max_trajectories=1`, tiny iteration counts. Good for smoke-testing the pipeline end-to-end on a new target.
- **Quick scan (~2-4h)**: `number_of_final_designs=5`, `max_trajectories=50`, leave iteration counts at upstream defaults. Enough designs to see if the target is tractable at all.
- **Production run (~24-72h)**: `number_of_final_designs=100`, no `max_trajectories` cap, default iteration counts. Matches the upstream paper's per-target budget.

In [ ]:
# Demonstrate `filter_overrides`: tighten Average_pLDDT and Average_i_pTM beyond
# the defaults. This cell only constructs the config (no GPU / no subprocess) so
# the notebook can be executed end-to-end within reasonable time budgets — to
# actually run, pass `strict_inputs` + `strict_config` to `run_bindcraft_design`.
strict_inputs = BindCraftInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    target_hotspot_residues="56",
    binder_lengths=(60, 70),
    binder_name="strict",
    number_of_final_designs=1,
)
strict_config = BindCraftConfig(
    max_trajectories=1,
    soft_iterations=10, temporary_iterations=5,
    hard_iterations=2, greedy_iterations=2,
    num_seqs=2, max_mpnn_sequences=1,
    seed=42,
    filter_overrides={
        "Average_pLDDT": {"threshold": 0.85, "higher": True},
        "Average_i_pTM": {"threshold": 0.6, "higher": True},
    },
)

print("Strict config:")
print(f"  filter_overrides = {strict_config.filter_overrides}")
# To actually run: strict_result = run_bindcraft_design(strict_inputs, strict_config)

In [ ]:
if result.designs:
    result.export("my_binder", file_format="pdb")  # writes a directory of PDBs + stats.json